# Rainfall Prediction: projeto completo de ciência de dados

Este notebook apresenta o projeto de ponta a ponta: objetivo, dados, cuidado com vazamento de informação, comparação de modelos, validação temporal, métricas e interpretação dos resultados.

A ideia é que ele funcione como uma versão visual e narrativa do projeto para recrutadores e entrevistas técnicas.

## 1. Objetivo do projeto

O objetivo é prever se choverá amanhã usando variáveis meteorológicas disponíveis hoje, como temperatura, umidade, pressão atmosférica, vento e chuva no dia atual.

Esse tipo de previsão pode apoiar decisões em logística, agricultura, construção civil, transporte e gestão de risco climático.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "database"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

pd.set_option("display.max_columns", 80)

## 2. Carregamento dos dados

O dataset usado é o `weatherAUS`, com observações meteorológicas de várias localidades da Austrália.

In [ ]:
raw_data = pd.read_csv(DATA_PATH, na_values=["NA"])

print(f"Linhas: {raw_data.shape[0]:,}")
print(f"Colunas: {raw_data.shape[1]:,}")
raw_data.head()

## 3. Variável alvo

A variável alvo é `RainTomorrow`, que indica se choveu no dia seguinte.

Como é um problema de classificação binária, as classes são:

- `No`: não chove amanhã.
- `Yes`: chove amanhã.

In [ ]:
target_distribution = (
    raw_data["RainTomorrow"]
    .value_counts(dropna=False)
    .rename_axis("RainTomorrow")
    .reset_index(name="count")
)
target_distribution["percentage"] = target_distribution["count"] / target_distribution["count"].sum()
target_distribution

## 4. O ponto mais importante: vazamento de dados

O dataset original contém a coluna `RISK_MM`, que representa a quantidade de chuva do dia seguinte.

Essa coluna está diretamente ligada ao alvo `RainTomorrow`. Se ela for usada no treino, o modelo aprende com uma informação do futuro e a avaliação fica artificialmente alta.

Por isso, `RISK_MM` é removida antes da modelagem. Essa decisão torna o resultado mais honesto e mais próximo de um cenário real.

In [ ]:
leakage_preview = raw_data[["RISK_MM", "RainTomorrow"]].dropna().head(10)
leakage_preview

## 5. Pipeline implementado

O projeto foi reorganizado em módulos Python reutilizáveis:

- `src/data.py`: carregamento e preparação dos dados.
- `src/features.py`: imputação, padronização e one-hot encoding.
- `src/models.py`: modelos candidatos.
- `src/evaluate.py`: métricas e gráficos.
- `src/train.py`: treinamento, avaliação e geração de artefatos.

O pipeline compara um baseline simples com modelos mais fortes, mantendo a avaliação reprodutível.

In [ ]:
from src.data import prepare_dataset

x, y = prepare_dataset(raw_data)

print("RISK_MM nas features?", "RISK_MM" in x.columns)
print("RainTomorrow nas features?", "RainTomorrow" in x.columns)
print(f"Total de features após preparação inicial: {x.shape[1]}")
x.head()

## 6. Comparação de modelos

Foram avaliados três modelos:

- `DummyClassifier`: baseline mínimo.
- `Logistic Regression`: modelo linear interpretável.
- `Random Forest`: modelo não linear, capaz de capturar interações entre variáveis meteorológicas.

As métricas abaixo foram geradas pelo script `python -m src.train`.

In [ ]:
metrics = pd.read_csv(REPORTS_DIR / "metrics.csv")
metrics

## 7. Validação temporal

Além da divisão estratificada tradicional, o projeto inclui uma validação temporal: treina em dados mais antigos e testa em dados mais recentes.

Esse teste é mais próximo do uso real de um modelo preditivo, porque simula uma previsão em períodos futuros.

In [ ]:
temporal_metrics = pd.read_csv(REPORTS_DIR / "temporal_metrics.csv")
temporal_metrics

## 8. Matriz de confusão

A matriz de confusão mostra os acertos e erros do melhor modelo em termos de dias com chuva e sem chuva.

![Matriz de confusão](../reports/figures/confusion_matrix.png)

## 9. Curva ROC

A curva ROC mostra a capacidade do modelo de separar as classes em diferentes limiares de decisão.

![Curva ROC](../reports/figures/roc_curve.png)

## 10. Curva Precision-Recall

Essa curva é especialmente útil quando existe desbalanceamento entre as classes. No caso de chuva, ela ajuda a entender o trade-off entre detectar mais dias chuvosos e evitar alarmes falsos.

![Curva Precision-Recall](../reports/figures/precision_recall_curve.png)

## 11. Importância das variáveis

A importância de variáveis ajuda a explicar quais sinais meteorológicos mais influenciaram o modelo.

![Importância de variáveis](../reports/figures/feature_importance.png)

In [ ]:
feature_importance = pd.read_csv(REPORTS_DIR / "feature_importance.csv")
feature_importance.head(15)

## 12. Principais conclusões

- A acurácia original acima de 98% era inflada por vazamento de dados.
- Após remover `RISK_MM`, os resultados ficaram mais realistas e confiáveis.
- A Random Forest teve o melhor equilíbrio geral entre `precision`, `recall`, `F1` e `ROC-AUC`.
- A validação temporal confirma que o modelo mantém desempenho razoável em dados mais recentes.
- Para um problema de chuva, `recall` é uma métrica importante porque mede a capacidade de detectar eventos chuvosos.

## 13. Como reproduzir

Para treinar novamente os modelos e atualizar os relatórios:

```bash
python -m src.train
```

Para abrir a demo interativa:

```bash
streamlit run app.py
```

Para rodar os testes:

```bash
python -m unittest discover
```